# MicroCapture — Book Page Pipeline v2 (classical CV)

**Phase 0: batch harness + baseline.**

This notebook replaces the single-image `boundary_prototype.ipynb` workflow. That
notebook processed **one uploaded image**, which is how its thresholds ended up
tuned to one photo and failing on the next.

Everything here runs over **all 130 images** in `Dataset-Training/output/`, and every
stage shows:

1. a **contact sheet of all 130** with the stage's overlay drawn on each,
2. a **histogram** of the stage's key metric,
3. the **auto-flagged failures, shown large**, with the reason printed,
4. the metric **broken out by content class**.

## Why content class matters

Measured on this corpus, the `|Gx|` column profile the old pipeline traced is **not
separable from page content**:

| content class | n | peak/interior ratio | rival columns |
|---|---|---|---|
| DIAGRAM/SPARSE | 32 | **4.82** | 53 |
| MIXED | 44 | 3.63 | 55 |
| TEXT-HEAVY | 24 | **3.00** | **196** |
| IMAGE-HEAVY | 26 | **2.92** | 122 |
| EMPTY/BLANK | 4 | 7.97 | 89 |

On image 43 the text body reaches `|Gx|` 80–95 while the true page edge peaks at ~95 —
**644 columns rival the true edge**. No threshold separates them. This is the measured
root cause of the "behaves differently for text-heavy / diagram-heavy / image-heavy"
behaviour, and it drives the v2 design: **stages key on region properties (paper
brightness), which barely move when the printed content changes, not on gradient
strength.**

In [ ]:
import os, glob, time, json, math
from dataclasses import dataclass, field
from typing import Optional, List, Tuple

import cv2
import numpy as np
import matplotlib.pyplot as plt

DATA = os.path.expanduser(
    '~/Micrographics/MicroCapture/Dataset-Training/output')
FILES = sorted(glob.glob(os.path.join(DATA, '*.jpg')))
print(f'{len(FILES)} images found in {DATA}')
im0 = cv2.imread(FILES[0])
print('native resolution:', im0.shape[1], 'x', im0.shape[0])

### Configuration and shared helpers

In [ ]:
# ─────────────────────────── config ───────────────────────────

# All geometry work happens at this width.  The captures are 6316x4211 (26MP);
# tracing at full res is ~16x the pixels for no accuracy gain, since the features
# we key on (page edges, gutter, fingers) are tens of pixels wide.  Final warps
# are applied at full resolution by scaling the geometry back up.
WORK_W = 1600

# Corner patch size (px, at WORK_W) used to sample the copy-stand background level.
BG_PATCH = 40


def to_work(img: np.ndarray, work_w: int = WORK_W) -> Tuple[np.ndarray, float]:
    """Downscale to working width. Returns (image, scale) where
    scale = work_w / original_w, so geometry can be mapped back up."""
    h, w = img.shape[:2]
    if w <= work_w:
        return img.copy(), 1.0
    s = work_w / w
    return cv2.resize(img, (work_w, int(round(h * s))), interpolation=cv2.INTER_AREA), s

### Display helpers

Every stage below shows **all 130** images: a contact sheet with the
stage's overlay, a metric histogram, the auto-flagged failures large with reasons, and the
metric by content class. No stage is ever judged on a single picture.

In [ ]:
import math
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec



def _rgb(img):
    if img.ndim == 2:
        return cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def thumb(img, width=260):
    h, w = img.shape[:2]
    s = width / float(w)
    return cv2.resize(img, (width, max(1, int(round(h * s)))),
                      interpolation=cv2.INTER_AREA)


def grid(images, titles=None, cols=6, width=260, title="",
         flags=None, figsize_scale=1.0):
    """Contact sheet of EVERY image passed in.

    flags: optional list of bools; flagged tiles get a red border and a red
    title, so problem cases stand out while scrolling a 130-tile sheet.
    """
    n = len(images)
    if n == 0:
        print(f"[{title}] nothing to show")
        return
    rows = math.ceil(n / cols)
    fig_w = cols * 2.6 * figsize_scale
    fig_h = rows * 2.1 * figsize_scale
    fig, axes = plt.subplots(rows, cols, figsize=(fig_w, fig_h))
    axes = np.atleast_1d(axes).ravel()
    for i, ax in enumerate(axes):
        ax.axis('off')
        if i >= n:
            continue
        im = thumb(images[i], width)
        bad = bool(flags[i]) if flags is not None else False
        if bad:
            im = cv2.copyMakeBorder(im, 6, 6, 6, 6, cv2.BORDER_CONSTANT,
                                    value=(0, 0, 255))
        ax.imshow(_rgb(im))
        if titles is not None:
            ax.set_title(titles[i], fontsize=7,
                         color=('red' if bad else 'black'))
    if title:
        fig.suptitle(title, fontsize=13, y=1.002)
    plt.tight_layout()
    plt.show()


def show_flagged(images, titles, flags, reasons=None, cols=3, width=520,
                 title="FLAGGED", max_show=24):
    """Show only the flagged images, large enough to actually diagnose."""
    idx = [i for i, f in enumerate(flags) if f]
    if not idx:
        print(f"[{title}] none flagged -- all clear")
        return
    print(f"[{title}] {len(idx)} flagged: {[titles[i] for i in idx][:40]}")
    idx = idx[:max_show]
    sel = [images[i] for i in idx]
    tt = []
    for i in idx:
        t = titles[i]
        if reasons is not None and reasons[i]:
            t += f"\n{reasons[i]}"
        tt.append(t)
    grid(sel, tt, cols=cols, width=width, title=title, figsize_scale=1.7)


def hist(values, title="", xlabel="", bins=30, vlines=None):
    values = np.asarray(values, dtype=float)
    plt.figure(figsize=(11, 3.2))
    plt.hist(values, bins=bins, color='#4878a8', edgecolor='white')
    if vlines:
        for v, lab, c in vlines:
            plt.axvline(v, color=c, ls='--', lw=1.4, label=lab)
        plt.legend(fontsize=8)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


def by_class(values, labels, title="", ylabel="", logy=False):
    """Box + strip plot of a metric grouped by content class.

    This is the plot that answers the operator's core concern directly: does this
    stage behave differently for text-heavy vs diagram-heavy vs image-heavy vs
    blank pages?
    """
    values = np.asarray(values, dtype=float)
    labels = list(labels)
    groups, names = [], []
    for c in CLASS_ORDER:
        v = values[[i for i, l in enumerate(labels) if l == c]]
        if len(v):
            groups.append(v)
            names.append(f"{c}\n(n={len(v)})")
    if not groups:
        return
    plt.figure(figsize=(11, 3.8))
    # matplotlib renamed boxplot's `labels` to `tick_labels` in 3.9; support both
    # so this runs on the operator's environment and on Kaggle alike.
    try:
        bp = plt.boxplot(groups, tick_labels=names, showfliers=False,
                         patch_artist=True)
    except TypeError:
        bp = plt.boxplot(groups, labels=names, showfliers=False,
                         patch_artist=True)
    for p in bp['boxes']:
        p.set_facecolor('#cfe0f0')
    for i, v in enumerate(groups):
        x = np.random.normal(i + 1, 0.055, len(v))
        plt.plot(x, v, '.', color='#c0392b', markersize=5, alpha=0.75)
    if logy:
        plt.yscale('log')
    plt.title(title)
    plt.ylabel(ylabel)
    plt.grid(alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()


def scorecard(labels, ok_flags, stage_name):
    """Pass rate per content class for one stage -- printed as a table."""
    labels = list(labels)
    ok = np.asarray(ok_flags, dtype=bool)
    print(f"\n=== {stage_name}: pass rate by content class ===")
    print(f"{'class':<16}{'n':>5}{'pass':>7}{'fail':>7}{'rate':>9}")
    total_ok = 0
    for c in CLASS_ORDER:
        ix = [i for i, l in enumerate(labels) if l == c]
        if not ix:
            continue
        k = int(ok[ix].sum())
        total_ok += k
        print(f"{c:<16}{len(ix):>5}{k:>7}{len(ix)-k:>7}{k/len(ix):>8.0%}")
    print(f"{'ALL':<16}{len(labels):>5}{total_ok:>7}"
          f"{len(labels)-total_ok:>7}{total_ok/max(len(labels),1):>8.0%}")


def overlay_mask(img, mask, color=(0, 0, 255), alpha=0.45):
    """Tint a binary mask over an image for visual checking."""
    out = img.copy()
    if mask is None:
        return out
    m = mask > 0
    layer = np.zeros_like(out)
    layer[:] = color
    out[m] = cv2.addWeighted(out, 1 - alpha, layer, alpha, 0)[m]
    return out


def draw_bbox(img, bbox, color=(0, 255, 0), t=3):
    out = img.copy()
    x, y, w, h = bbox
    cv2.rectangle(out, (x, y), (x + w, y + h), color, t)
    return out

## 1. Stage code — the whole pipeline

Every algorithm lives here, in the notebook. Edit a function and re-run this cell, then
re-run the driver below to see the effect across all 130 images. Nothing is imported from
an external module.

### Stage 1 — deskew
Ported from `bookcurve (8).ipynb`: Otsu -> largest contour -> minAreaRect -> rotate. Keys on
the book SILHOUETTE, not page content, so it is content-invariant by construction.

In [ ]:
# ─────────────────────── stage 1: deskew ───────────────────────

@dataclass
class DeskewResult:
    angle: float                 # degrees applied (positive = counter-clockwise)
    rotated: np.ndarray          # deskewed image
    rect: tuple                  # cv2.minAreaRect of the book silhouette
    silhouette_frac: float       # book area / frame area -- sanity metric
    note: str = ""


def _book_silhouette(gray: np.ndarray) -> Optional[np.ndarray]:
    """Largest external contour of the book against the copy stand.

    Otsu is the right tool here and is content-invariant in the way we need: it
    separates the (bright, large) book from the (dark, large) stand.  What is
    printed on the page shifts the page's own histogram mass around a little but
    does not move the book/stand split, because paper stays far brighter than the
    stand even where it is densely printed.  Verified on all 130 (bg median 34,
    page median 221).
    """
    _, th = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    th = cv2.morphologyEx(th, cv2.MORPH_CLOSE, np.ones((25, 25), np.uint8))
    cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    return max(cnts, key=cv2.contourArea)


def _normalize_angle(angle: float) -> float:
    """Map minAreaRect's angle into (-45, 45].

    NOTE this is a SKEW corrector, not an orientation corrector: it cannot tell
    0 from 90 degrees.  A portrait page shot in landscape is left alone.  That is
    correct for this rig (fixed overhead camera, book always roughly landscape)
    but must not be mistaken for full orientation handling.
    """
    angle = angle % 90
    if angle > 45:
        angle -= 90
    return angle


def rotate_bound(img: np.ndarray, angle: float, border=(0, 0, 0)) -> np.ndarray:
    """Rotate about the centre, expanding the canvas so nothing is clipped."""
    h, w = img.shape[:2]
    cx, cy = w // 2, h // 2
    M = cv2.getRotationMatrix2D((cx, cy), angle, 1.0)
    cos, sin = abs(M[0, 0]), abs(M[0, 1])
    nW = int((h * sin) + (w * cos))
    nH = int((h * cos) + (w * sin))
    M[0, 2] += (nW / 2) - cx
    M[1, 2] += (nH / 2) - cy
    return cv2.warpAffine(img, M, (nW, nH), borderValue=border,
                          flags=cv2.INTER_LINEAR)


def deskew(img: np.ndarray) -> DeskewResult:
    """Stage 1 -- rotate the book upright.

    Ported from the operator's own `bookcurve (8).ipynb`, which ran 130/130 with
    zero errors (angle range -16.66..+15.33 deg, mean abs 2.68).  Why it beats
    the app's current text-line/Hough deskew: it keys on the BOOK SILHOUETTE, not
    on page content, so a diagram-heavy or blank page deskews exactly as well as
    a text page.  That is the same content-invariance principle as rule 2 above.

    Runs FIRST because every later stage traces along rows/columns and therefore
    assumes the book is axis-aligned.
    """
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    c = _book_silhouette(gray)
    if c is None:
        return DeskewResult(0.0, img.copy(), ((0, 0), (0, 0), 0), 0.0,
                            "no silhouette contour -- left unrotated")
    rect = cv2.minAreaRect(c)
    angle = _normalize_angle(rect[-1])
    frac = cv2.contourArea(c) / float(img.shape[0] * img.shape[1])
    rotated = rotate_bound(img, angle)
    note = ""
    if frac < 0.10:
        note = f"silhouette only {frac:.1%} of frame -- suspicious"
    elif frac > 0.92:
        note = f"silhouette {frac:.1%} of frame -- book may exceed frame"
    return DeskewResult(angle, rotated, rect, frac, note)

### Stage 2 — paper-region segmentation
The core change vs the old pipeline: segment paper as a REGION instead of tracing the
strongest gradient, which page content defeats.

In [ ]:
# ──────────────── stage 2: paper-region segmentation ────────────────

@dataclass
class PaperResult:
    mask: np.ndarray             # uint8 0/255, the paper region (both pages + block)
    bg_level: float
    paper_level: float
    contrast: float
    area_frac: float
    bbox: Tuple[int, int, int, int]   # x, y, w, h
    # Fraction of the mask's own minAreaRect that the mask fills.  An open book is
    # very nearly a filled rectangle (measured 0.81-0.95); a book with a wall blob
    # or a hand fused onto it is not.  This metric exists because area_frac ALONE
    # passed ~40 images whose masks were visibly contaminated -- the blob kept the
    # area inside the accepted window.  Shape catches what area cannot.
    rect_fill: float = 0.0
    found: bool = True
    note: str = ""


def segment_paper(img: np.ndarray) -> PaperResult:
    """Stage 2 -- segment paper from the copy stand.

    THIS IS THE KEY DESIGN CHANGE vs the old pipeline.  Instead of asking "where
    is the strongest vertical gradient" (which text defeats -- see module
    docstring), we ask "which pixels are paper".  Paper is bright and, crucially,
    stays bright where it is printed on: dense text lowers a neighbourhood's mean
    by a few tens of levels, while the paper-to-stand step is ~185 levels on this
    corpus.  So a brightness-based region test is nearly content-invariant, which
    is exactly the property the gradient test lacked.

    Robustness details that matter on the real corpus:
      * Threshold is derived from the image's OWN bg/paper levels rather than a
        fixed constant, because lighting falls off across the stand (measured:
        bg 12..60 across the 130).
      * Morphological close then open: close bridges text/gutter shadow so the
        page reads as one region; open removes stand specks (the rig's bright
        bolts) that would otherwise seed false regions.
      * Largest connected component only -- the book is one object.
    """
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape

    p = BG_PATCH
    corners = np.concatenate([
        gray[:p, :p].ravel(), gray[:p, -p:].ravel(),
        gray[-p:, :p].ravel(), gray[-p:, -p:].ravel()])
    bg = float(np.median(corners))

    # Paper level: the bright mode of the image.  Use a high percentile rather
    # than Otsu's class mean so heavy ink doesn't drag the estimate down.
    paper = float(np.percentile(gray, 90))
    contrast = paper - bg
    note = ""

    # Threshold is anchored on the PAPER level, not on Otsu's split and not on an
    # offset from the corner background.  Both of those were tried on the real
    # corpus and both failed, in opposite directions:
    #
    #   * bg + k*contrast  -- images 90-121 are shot with a lit wall behind the
    #     stand.  Corner patches read ~47 (stand) while the frame median is ~150
    #     (wall), so a corner-anchored threshold sits under the wall and the wall
    #     floods in: paper fractions of 0.77-0.98 where the book covers ~half.
    #
    #   * Otsu -- better, but still lands at 126-148 on those same images while
    #     the wall itself sits at 90-118.  Visual check of all 130 (not the area
    #     metric, which happily passed them) showed a wall blob fused to the book
    #     in ~40 masks, and because it is CONNECTED to the book, "largest
    #     component" cannot drop it.
    #
    # Paper, however, is consistently ~220-240 while the wall is ~90-118: a gap of
    # 116+ levels on every image measured.  So key the threshold on the paper mode
    # and cut well below it but far above the wall.  This is the same
    # content-invariance argument as the module docstring: ink lowers a
    # neighbourhood mean by tens of levels, nothing like this gap.
    otsu_t = float(cv2.threshold(gray, 0, 255,
                                 cv2.THRESH_BINARY + cv2.THRESH_OTSU)[0])

    if contrast < 25:
        # Black cover / very dark material (imgs 17, 18): brightness alone cannot
        # separate object from stand.  Otsu is all we have; flag it.
        _, th = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        note = f"low contrast ({contrast:.0f}) -- dark cover, Otsu only"
    else:
        # 0.62 of the way from stand to paper.  Low enough to keep shadowed paper
        # near the gutter and the darker fore-edge block; high enough to exclude a
        # lit wall.  Also require the cut to clear Otsu, so on a plain dark-stand
        # image (where Otsu is already correct) we never threshold lower than it.
        t = max(bg + contrast * 0.62, otsu_t)
        _, th = cv2.threshold(gray, t, 255, cv2.THRESH_BINARY)

    th = cv2.morphologyEx(th, cv2.MORPH_CLOSE, np.ones((31, 31), np.uint8))
    th = cv2.morphologyEx(th, cv2.MORPH_OPEN, np.ones((15, 15), np.uint8))

    n, lab, stats, _ = cv2.connectedComponentsWithStats(th, 8)
    if n <= 1:
        return PaperResult(np.zeros_like(gray), bg, paper, contrast, 0.0,
                           (0, 0, 0, 0), 0.0, False, "no paper region found")
    k = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    mask = np.where(lab == k, 255, 0).astype(np.uint8)

    # Fill interior holes (dark photos on the page must not punch through).
    ff = mask.copy()
    pad = np.zeros((mask.shape[0] + 2, mask.shape[1] + 2), np.uint8)
    cv2.floodFill(ff, pad, (0, 0), 255)
    mask = mask | cv2.bitwise_not(ff)

    # ---- trim side blobs by column/row occupancy -------------------------
    # Raising the threshold shrank the lit-wall blob on images ~90-130 but did not
    # remove it: the wall is bright enough in places to survive any brightness cut
    # that still keeps shadowed paper, it is CONNECTED to the book so component
    # selection cannot drop it, and it is convex enough that rect_fill only caught
    # one of them.  So separate it structurally instead.
    #
    # The signature is unambiguous in the mask's own column-occupancy profile: a
    # wall/hand blob occupies ~0.17-0.19 of the column height, while a real book
    # column occupies ~0.7+.  Keep only the central run of columns (and then rows)
    # that clear a fraction of the profile's own peak -- so this adapts per image
    # rather than assuming a fixed page height.  A book is one solid horizontal
    # run by construction, so taking the run CONTAINING THE PEAK column can never
    # split a genuine spread.
    def _trim(m: np.ndarray, axis: int, keep_frac: float = 0.45) -> np.ndarray:
        occ = (m > 0).sum(axis=axis).astype(np.float32)
        if occ.max() <= 0:
            return m
        good = occ >= occ.max() * keep_frac
        if not good.any():
            return m
        peak = int(np.argmax(occ))
        lo = peak
        while lo > 0 and good[lo - 1]:
            lo -= 1
        hi = peak
        while hi < len(good) - 1 and good[hi + 1]:
            hi += 1
        out = np.zeros_like(m)
        if axis == 0:      # occ indexed by column -> keep columns lo..hi
            out[:, lo:hi + 1] = m[:, lo:hi + 1]
        else:              # occ indexed by row
            out[lo:hi + 1, :] = m[lo:hi + 1, :]
        return out

    trimmed = _trim(_trim(mask, axis=0), axis=1)
    if (trimmed > 0).sum() > 0.25 * max((mask > 0).sum(), 1):
        # Only accept the trim if it kept the bulk of the region; otherwise the
        # occupancy profile was not book-shaped and we keep the untrimmed mask
        # rather than silently discarding most of the page.
        mask = trimmed
    else:
        note = ((note + "; " if note else "")
                + "occupancy trim rejected (would remove >75% of mask)")

    x, y, bw, bh = cv2.boundingRect(mask)
    frac = float((mask > 0).mean())

    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if cnts:
        c = max(cnts, key=cv2.contourArea)
        (_, _), (rw, rh), _ = cv2.minAreaRect(c)
        rect_area = rw * rh
        rect_fill = float(cv2.contourArea(c) / rect_area) if rect_area > 0 else 0.0
    else:
        rect_fill = 0.0

    if frac < 0.08:
        note = (note + "; " if note else "") + f"paper only {frac:.1%} of frame"
    if rect_fill and rect_fill < 0.75:
        note = ((note + "; " if note else "")
                + f"mask fills only {rect_fill:.0%} of its rect -- blob attached?")
    return PaperResult(mask, bg, paper, contrast, frac, (x, y, bw, bh),
                       rect_fill, True, note)

### Scope gate + content classifier
Rejects closed books; keeps single sheets. The classifier is diagnostic only and never
routes the algorithm.

In [ ]:
# ───────────── scope gate: is this a capture we should process? ─────────────

@dataclass
class ScopeResult:
    in_scope: bool
    kind: str                    # 'spread' | 'single' | 'closed-book'
    white_frac: float            # fraction of the paper region that is near-white
    aspect: float                # region bbox width/height
    reason: str = ""


def check_scope(img: np.ndarray, paper: PaperResult) -> ScopeResult:
    """Decide whether this capture is something the pipeline should process.

    OUT OF SCOPE: closed books (a photographed cover).  The operator confirmed
    these are not to be processed -- there is no page to flatten, no gutter, and
    no split.  In the corpus these are #17, 18, 30, 31, 32, 33.

    IN SCOPE: open spreads AND single-page documents (flyers, magazine covers,
    brochures -- #55-61, 68-73).  Single sheets are kept because they are
    genuinely EASIER than spreads, not harder: measured rect_fill 0.88-0.997
    against 0.95-0.99 for spreads, i.e. they segment at least as cleanly.  They
    simply take the no-gutter path -- boundary and crop, no split.

    The discriminator is the fraction of the detected region that is near-white
    PAPER.  This works because it keys on what the material physically IS rather
    than on its size or brightness, both of which were tried and failed:

      * area fraction    -- #43 (a valid spread) sits at 0.226, right among the
                            closed books; no threshold separates them.
      * region brightness-- #71/#73 (valid single sheets) have median 102-104,
                            BELOW several closed books at 112-153.

    White fraction separates cleanly: closed books 0.010-0.069, valid pages
    0.157-0.924.  That is a 2.3x gap at the boundary.

    The one genuine ambiguity is a full-bleed colour spread with almost no white
    margin (#115, white 0.093).  Aspect resolves it: an open spread is landscape
    (measured 1.14-1.59) while a closed book or portrait sheet is not, so a wide
    region is accepted as a spread regardless of how little white it shows.
    """
    white_thresh = 200
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    m = paper.mask > 0
    if m.sum() < 100:
        return ScopeResult(False, 'closed-book', 0.0, 0.0,
                           "no paper region detected")

    vals = gray[m]
    white = float((vals > white_thresh).mean())
    x, y, bw, bh = paper.bbox
    aspect = bw / float(bh) if bh > 0 else 0.0

    # A clearly landscape region is an open spread: both pages side by side.  Even
    # a full-bleed colour spread with no white margin is in scope.
    #
    # The aspect band is bounded ABOVE as well as below.  On a dark cover the
    # brightness segmentation catches only the bright title strip, which is a thin
    # sliver -- #17 and #18 measure aspect 3.77 and 2.48 that way and would sail
    # through an unbounded "wide means spread" test.  A real open spread measures
    # 1.14-1.59 across this corpus, so anything beyond 2.0 is not a spread, it is a
    # fragment of something else.
    if 1.10 <= aspect <= 2.00:
        return ScopeResult(True, 'spread', white, aspect, "")

    if aspect > 2.00:
        return ScopeResult(False, 'closed-book', white, aspect,
                           f"region aspect {aspect:.2f} is a sliver, not a page "
                           f"-- partial detection on a dark cover?")

    if white < 0.12:
        return ScopeResult(False, 'closed-book', white, aspect,
                           f"only {white:.1%} of region is paper-white and "
                           f"aspect {aspect:.2f} is not a spread -- closed book?")

    return ScopeResult(True, 'single', white, aspect, "")



# ───────────────── content classification (diagnostic) ─────────────────

@dataclass
class ContentStats:
    ink: float
    edge_density: float
    textiness: float
    colorfulness: float
    big_blob: float
    label: str


def classify_content(img: np.ndarray, paper_mask: np.ndarray) -> ContentStats:
    """Diagnostic ONLY -- never routes the algorithm.

    We deliberately do not branch the pipeline on content type: that would mean
    five code paths, five threshold sets, and a classifier whose own errors land
    exactly on the hard images.  Instead one content-invariant algorithm runs for
    everything and this label is used to GROUP the scorecard, so a change that
    helps text-heavy pages while breaking image-heavy ones is immediately visible
    instead of averaging out.
    """
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    ys, xs = np.where(paper_mask > 0)
    if len(xs) < 1000:
        roi = gray
        roi_sat = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)[:, :, 1]
    else:
        # Erode inward so the block/stand edge doesn't pollute content stats.
        er = cv2.erode(paper_mask, np.ones((41, 41), np.uint8))
        ys, xs = np.where(er > 0)
        if len(xs) < 1000:
            er = paper_mask
            ys, xs = np.where(er > 0)
        y0, y1, x0, x1 = ys.min(), ys.max(), xs.min(), xs.max()
        roi = gray[y0:y1, x0:x1]
        roi_sat = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)[y0:y1, x0:x1, 1]

    paper_level = float(np.median(roi))
    ink = float((roi < paper_level - 40).mean())

    edges = cv2.Canny(roi, 50, 150)
    edge_density = float(edges.mean() / 255.0)

    sob_y = cv2.Sobel(roi, cv2.CV_32F, 0, 1, ksize=3)
    rp = np.abs(sob_y).mean(axis=1)
    rp = rp - rp.mean()
    if rp.std() > 0 and len(rp) > 70:
        ac = np.correlate(rp, rp, 'full')[len(rp) - 1:]
        ac = ac / (ac[0] + 1e-9)
        seg = ac[8:60]
        textiness = float(seg.max()) if seg.size else 0.0
    else:
        textiness = 0.0

    colorfulness = float(roi_sat.mean())

    dark = (roi < paper_level - 60).astype(np.uint8)
    dark = cv2.morphologyEx(dark, cv2.MORPH_OPEN, np.ones((9, 9), np.uint8))
    nb, _, st, _ = cv2.connectedComponentsWithStats(dark, 8)
    big_blob = float(st[1:, cv2.CC_STAT_AREA].max() / roi.size) if nb > 1 else 0.0

    if ink < 0.04:
        label = 'EMPTY/BLANK'
    elif colorfulness > 45 or big_blob > 0.14:
        label = 'IMAGE-HEAVY'
    elif textiness > 0.60 and ink > 0.12:
        label = 'TEXT-HEAVY'
    elif big_blob > 0.05 or edge_density < 0.09:
        label = 'DIAGRAM/SPARSE'
    else:
        label = 'MIXED'

    return ContentStats(ink, edge_density, textiness, colorfulness, big_blob, label)


CLASS_ORDER = ['TEXT-HEAVY', 'DIAGRAM/SPARSE', 'IMAGE-HEAVY', 'MIXED', 'EMPTY/BLANK']

### Stage 3 — front-page boundary and gutter
Bounds the printed pages, excluding the fore-edge block. Keyed on per-column mean
brightness, which content barely perturbs.

In [ ]:
# ───────────── stage 3: front-page boundary + gutter ─────────────

@dataclass
class PageBoundary:
    """Vertical structure of the capture, in deskewed working-image coordinates.

    left/right are the OUTER edges of the printed pages -- the operator's "front
    pages, not side pages" requirement.  The fore-edge block (the stack of
    remaining leaves, which is paper and therefore inside the paper mask) sits
    OUTSIDE these and is excluded.
    """
    left: int                    # x of the left page's outer edge
    right: int                   # x of the right page's outer edge (exclusive)
    top: int
    bottom: int
    gutter: Optional[int]        # x of the spine, or None for a single sheet
    kind: str                    # 'spread' | 'single'
    block_left: int              # px of fore-edge block / blob trimmed on the left
    block_right: int             # px trimmed on the right
    found: bool = True
    profile: Optional[np.ndarray] = None   # brightness profile, for plots
    page_level: float = 0.0
    note: str = ""


def _column_brightness(gray, mask, bbox):
    """Mean brightness per column over the vertical middle of the book.

    The middle band is used rather than the full height because the top and
    bottom of a spread curve away, and are where fingers usually sit; the middle
    is the most reliable cross-section.  Masked-out pixels are excluded so the
    dark stand cannot drag a column's mean down.
    """
    x, y, bw, bh = bbox
    y0, y1 = y + int(bh * 0.30), y + int(bh * 0.70)
    band = gray[y0:y1, x:x + bw].astype(np.float32)
    mband = (mask[y0:y1, x:x + bw] > 0).astype(np.float32)
    denom = np.maximum(mband.sum(axis=0), 1.0)
    prof = (band * mband).sum(axis=0) / denom
    prof[mband.sum(axis=0) < (y1 - y0) * 0.25] = 0.0   # too little paper here
    return prof


def _smooth(a, k):
    if k < 3:
        return a
    k = int(k) | 1
    return np.convolve(a, np.ones(k) / k, mode='same')


def _longest_run(flags):
    """(lo, hi) of the longest True run; hi exclusive.  (0, 0) if none."""
    best_lo, best_hi, lo = 0, 0, None
    for i, v in enumerate(np.append(np.asarray(flags, bool), False)):
        if v and lo is None:
            lo = i
        elif not v and lo is not None:
            if i - lo > best_hi - best_lo:
                best_lo, best_hi = lo, i
            lo = None
    return best_lo, best_hi


def detect_boundary(img, paper, scope):
    """Stage 3 -- find the printed pages' outer edges and the gutter.

    Evidence is the per-column MEAN BRIGHTNESS of the paper region, not a
    gradient.  Measured on the corpus this profile has exactly the structure we
    need, and page content barely perturbs it:

      * the fore-edge block, and the lit-wall blob, read as a wide flat DARKER
        plateau (~140-155) ending in a sharp cliff up to page level (~215-230)
        -- clearly visible on #124 and #109;
      * the gutter reads as a NARROW deep dip inside the page region (#43 at
        x~430, #124 at x~730);
      * printed content is only a few levels of high-frequency noise on top of
        the page plateau -- which is precisely why this works where gradient
        tracing failed (text reaches the same |Gx| magnitude as the true edge,
        but it does not move the column MEAN).

    Page level is a high percentile so heavy ink does not drag it down.  The page
    span is the LONGEST sustained run of near-page-level columns: taking the
    longest run rather than the first is what rejects both the fore-edge block
    and the wall blob, since each is a separate and shorter run.
    """
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    x, y, bw, bh = paper.bbox
    if bw < 40 or bh < 40:
        return PageBoundary(x, x + bw, y, y + bh, None, scope.kind, 0, 0,
                            False, None, 0.0, "paper bbox too small")

    prof = _column_brightness(gray, paper.mask, paper.bbox)
    sm = _smooth(prof, max(5, bw // 120))
    nz = sm[sm > 0]
    if nz.size < 10:
        return PageBoundary(x, x + bw, y, y + bh, None, scope.kind, 0, 0,
                            False, prof, 0.0, "no lit columns")
    page_level = float(np.percentile(nz, 75))

    # A column counts as page when it is within this much of the page plateau.
    # The block/blob sits 60-80 levels below page level, so 0.88 separates them
    # while still tolerating the shading gradient across a curved page.
    on_page = sm > page_level * 0.88

    # CLOSE narrow gaps before taking the longest run.  The gutter is a deep but
    # NARROW spike that dips below the on-page threshold (measured: #124 at
    # x~950, #109 at x~930), and a plain longest-run then returns only ONE PAGE
    # of the spread -- the first version of this function did exactly that,
    # reporting left=957 right=1491 for #124, which is the right-hand page alone.
    # A gutter is tens of columns wide; the block plateau is hundreds.  Closing
    # gaps up to ~8% of the book width therefore bridges the spine without ever
    # bridging the block, so the span covers BOTH pages and the gutter can then be
    # located inside it.
    gap_close = max(5, int(bw * 0.08))
    closed = cv2.morphologyEx(on_page.astype(np.uint8).reshape(1, -1),
                              cv2.MORPH_CLOSE,
                              np.ones((1, gap_close), np.uint8)).ravel() > 0
    lo, hi = _longest_run(closed)
    if hi <= lo:
        return PageBoundary(x, x + bw, y, y + bh, None, scope.kind, 0, 0,
                            False, prof, page_level, "no sustained page run")

    # The brightness run gives the printed-page span, but ONLY where the page is
    # bright enough to read as "page level".  On ink-heavy material (full-bleed
    # photos, colour magazine spreads) the ink drags the column mean below the
    # threshold and the run stops short: measured 37 of 124 in-scope captures
    # under-detecting, every one of them with white fraction <= 0.75, some as
    # low as span 0.20 of the paper bbox.
    #
    # Stage 2's paper mask does not have that problem -- it segments paper vs
    # stand, which ink barely moves -- so the mask's own extent is the honest
    # outer bound.  Only trim inward from it where a genuinely DARKER PLATEAU (a
    # fore-edge block or wall blob) was found: that is a sustained low run at the
    # frame edge, not merely a dark page.  Trimming is capped so a dark page can
    # never lose more than a quarter of its width.
    max_trim = int(bw * 0.25)
    trim_lo = lo if lo <= max_trim else 0
    trim_hi = (bw - hi) if (bw - hi) <= max_trim else 0

    # Find the BLOCK->PAGE SHADOW LINE instead of testing for a dark plateau.
    #
    # The dark-plateau test this replaces was measured wrong and did nothing: on #1
    # the fore-edge block reads 216-221 against a page at 223, a five-level
    # difference, so the plateau test rejected every legitimate trim and left
    # block_left = block_right = 0 on most captures.  span = 1.00 then looked like
    # a pass while the green edges sat on the book block, not the printed page --
    # exactly the failure the operator identified.
    #
    # What IS present is a shadow line: where the printed page lifts off the block
    # it casts a narrow dark line, measured 38-55 levels deep on #19/#20/#29/#43.
    # It is not always present (#1 and #2 show only 4-6 levels, a book lying flat
    # enough that the top page sits directly on the block), so a missing shadow
    # means "no trim", not "trim anyway".
    def _shadow_trim(prof_slice, limit):
        """Deepest sustained dark line within `limit` px of an outer edge."""
        if limit < 12 or prof_slice.size < 12:
            return 0
        seg = prof_slice[:limit]
        k = int(np.argmin(seg))
        if k < 4 or k > limit - 4:
            return 0
        depth = page_level - float(seg[k])
        return k if depth > page_level * 0.10 else 0

    search = int(bw * 0.25)
    trim_lo = _shadow_trim(sm, min(search, bw - 1))
    trim_hi = _shadow_trim(sm[::-1], min(search, bw - 1))

    # A wall/hand blob is a wide DARK plateau rather than a narrow shadow line, so
    # keep the old occupancy-derived trim when it is larger -- that is what removes
    # the lit-wall blob on #109/#112/#117/#121-123.
    trim_lo = max(trim_lo, lo if lo <= max_trim else 0)
    trim_hi = max(trim_hi, (bw - hi) if (bw - hi) <= max_trim else 0)

    left, right = x + trim_lo, x + bw - trim_hi
    block_left, block_right = trim_lo, trim_hi
    lo, hi = trim_lo, bw - trim_hi

    # ---- gutter: the deepest narrow dip INSIDE the page span ---------------
    gutter, note = None, ""
    if scope.kind == 'spread':
        inner = sm[lo:hi]
        if inner.size > 60:
            # Ignore the outer 15% of the span: a page edge's own roll-off is not
            # a gutter, and a real spine sits well inside.
            m = int(inner.size * 0.15)
            core = inner[m:inner.size - m]
            k = int(np.argmin(core))
            depth = page_level - float(core[k])
            # 6% of page level (~13 levels).  Lowered from 10% after measuring the
            # real distribution: a flat-lying book has a genuine but shallow spine
            # shadow, and 10% found a gutter on only 14 of 108 spreads.
            if depth > page_level * 0.06:
                gutter = left + m + k
            else:
                note = (f"no gutter dip (deepest {depth:.0f}, "
                        f"need {page_level * 0.06:.0f})")

    return PageBoundary(left, right, y, y + bh, gutter, scope.kind,
                        block_left, block_right, True, prof, page_level, note)

### Stage 4 — finger mask and page-colour fill

In [ ]:
# ───────────── stage 4: finger mask + page-colour fill ─────────────

@dataclass
class FingerResult:
    mask: np.ndarray             # uint8 0/255, skin overlapping the page
    filled: np.ndarray           # image with the mask filled in page colour
    n_regions: int
    area_frac: float             # fraction of the page area covered
    over_text: bool              # any region sits on inked content, not margin
    note: str = ""


# YCrCb skin bounds.  Cr/Cb are chroma channels, so this is largely invariant to
# how brightly the hand is lit -- which matters because the same hand is lit very
# differently at the edge of the stand than over the page.
# Chroma alone CANNOT separate skin from this corpus's cream book paper.  Measured
# in Lab over hand-picked patches: skin a* mean 134.9 (min 128.8), paper a* mean
# 130.3 (max 138.7) -- the ranges OVERLAP, gap -9.8.  An earlier version relied on
# chroma plus a loose luma test and produced masks that traced the cream fore-edge
# block down the full page height (visible on #19-29) while catching only fingertips
# of real hands.
#
# LUMA is what actually separates them, decisively.  Measured across 12 captures
# with hands, hand luma sits at 0.43-0.57 of the page's own luma (mean ~0.49),
# while the fore-edge block sits at ~0.95 of page luma (216-221 against 223).  A
# page-relative ratio is used rather than an absolute level so it survives the
# lighting falloff across the stand.  Chroma is kept only as a secondary filter to
# exclude dark PRINT, which is neutral, from the dark-region candidates.
# Wide, permissive chroma window: it only has to exclude neutral PRINT (a
# dark photo or heavy text block is achromatic, Cr/Cb near 128) from the
# dark-region candidates.  Luma does the real separating.
SKIN_CR = (133, 180)
SKIN_CB = (70, 133)
# Skin is darker than the page it covers.  Measured on this rig, page sits ~215-240
# while a lit hand sits well below; this ratio rejects warm paper outright.
SKIN_MAX_LUMA_RATIO = 0.62
# A finger region larger than this fraction of the page is not a finger.
FINGER_MAX_AREA_FRAC = 0.22


def detect_fingers(img: np.ndarray, paper: PaperResult,
                   bound: 'PageBoundary') -> FingerResult:
    """Stage 4 -- find fingers over the page and fill them with page colour.

    The operator's spec is explicit: fill the finger region with the colour of the
    page.  That is deliberately NOT content reconstruction -- nothing is invented
    that could pass as real text.  A flat fill either looks right (on a margin,
    which is the dominant case: thumbs holding the book open at the page edges) or
    looks obviously like a patch, which is a safe failure.

    Two things make this reliable here that were not available to the old
    pipeline's skin detector:

      1. Stage 3 already knows where the printed page is, so we only consider skin
         INSIDE the page bounds.  A hand resting on the stand is irrelevant.
      2. A printed photograph of a person is the classic false positive.  It is
         rejected structurally: a real finger enters from outside the page, so its
         region must touch the page boundary.  A skin-toned region fully enclosed
         by page content never does, however finger-shaped it looks.

    The fill colour is sampled LOCALLY from a dilated ring of page pixels around
    each region, not as one global page colour: a curved page shades noticeably
    from gutter to fore-edge, so a single flat value would show as a mismatched
    rectangle.
    """
    h, w = img.shape[:2]
    page = np.zeros((h, w), np.uint8)
    page[bound.top:bound.bottom, bound.left:bound.right] = 255
    page = cv2.bitwise_and(page, (paper.mask > 0).astype(np.uint8) * 255)

    ycrcb = cv2.cvtColor(img, cv2.COLOR_BGR2YCrCb)
    cr, cb = ycrcb[:, :, 1], ycrcb[:, :, 2]
    gray0 = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    pp = gray0[page > 0]
    plevel = float(np.percentile(pp, 75)) if pp.size else 220.0
    skin = ((cr >= SKIN_CR[0]) & (cr <= SKIN_CR[1]) &
            (cb >= SKIN_CB[0]) & (cb <= SKIN_CB[1]) &
            (gray0 < plevel * SKIN_MAX_LUMA_RATIO) &
            # exclude neutral print: skin is chromatic, ink is not
            ((cr.astype(np.int16) - 128) + (128 - cb.astype(np.int16)) > 8)
            ).astype(np.uint8) * 255
    skin = cv2.morphologyEx(skin, cv2.MORPH_OPEN, np.ones((9, 9), np.uint8))
    skin = cv2.morphologyEx(skin, cv2.MORPH_CLOSE, np.ones((15, 15), np.uint8))

    # Detect skin over the WHOLE frame, then decide by CONNECTIVITY, not by
    # cropping to the page first.  Cropping first was wrong: a hand enters the
    # frame from OUTSIDE the page rect, so clipping to the page cut each finger
    # into a fragment whose bounding box no longer reached the page boundary, and
    # the touch test then rejected it.  That is why obvious hand captures (#13,
    # #23, #40) were being missed while only fingertips survived elsewhere.
    skin_on_page = skin

    page_area = float(max((page > 0).sum(), 1))
    keep = np.zeros((h, w), np.uint8)
    n, lab, stats, _ = cv2.connectedComponentsWithStats(skin_on_page, 8)
    kept = 0
    for k in range(1, n):
        area = stats[k, cv2.CC_STAT_AREA]
        if area < page_area * 0.002:          # noise / skin-toned print speckle
            continue
        x0, y0, ww, hh = (stats[k, cv2.CC_STAT_LEFT], stats[k, cv2.CC_STAT_TOP],
                          stats[k, cv2.CC_STAT_WIDTH], stats[k, cv2.CC_STAT_HEIGHT])
        # A real hand enters the frame from outside, so its region must reach the
        # FRAME edge.  A printed photograph of a person -- the classic false
        # positive -- is fully enclosed by the page and never does, however
        # finger-shaped it looks.  This is the structural test that replaces
        # tuning, and it is checked on the whole-frame component.
        m = 8
        reaches_frame = (x0 <= m or y0 <= m or
                         x0 + ww >= w - m or y0 + hh >= h - m)
        if not reaches_frame:
            continue
        # ...and it must actually overlap the page, or there is nothing to fill.
        comp = (lab == k)
        on_page_px = int((comp & (page > 0)).sum())
        if on_page_px < page_area * 0.0008:
            continue
        if on_page_px > page_area * FINGER_MAX_AREA_FRAC:
            continue          # too big to be a finger -- shadow band or dark page
        keep[comp & (page > 0)] = 255
        kept += 1

    if kept == 0:
        return FingerResult(keep, img.copy(), 0, 0.0, False, "no finger on page")

    keep = cv2.dilate(keep, np.ones((13, 13), np.uint8))   # cover the soft edge
    keep = cv2.bitwise_and(keep, page)

    # ---- does any region sit on inked content rather than blank margin? -----
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    ring = cv2.dilate(keep, np.ones((41, 41), np.uint8)) & ~keep & page
    page_px = gray[page > 0]
    page_level = float(np.percentile(page_px, 75)) if page_px.size else 220.0
    ring_px = gray[ring > 0]
    over_text = bool(ring_px.size and (ring_px < page_level * 0.75).mean() > 0.18)

    # ---- fill with LOCAL page colour ---------------------------------------
    filled = img.copy()
    # Blur the image heavily with the finger pixels excluded, so each filled pixel
    # takes the colour of nearby PAGE, following the page's own shading gradient.
    src = img.copy()
    src[keep > 0] = 0
    valid = (page > 0) & (keep == 0)
    vf = valid.astype(np.float32)
    k = 121
    num = cv2.blur(src.astype(np.float32) * vf[..., None], (k, k))
    den = cv2.blur(vf, (k, k))[..., None]
    local = num / np.maximum(den, 1e-3)
    filled[keep > 0] = np.clip(local[keep > 0], 0, 255).astype(np.uint8)

    frac = float((keep > 0).sum() / page_area)
    note = "finger overlaps inked content -- fill will erase it" if over_text else ""
    return FingerResult(keep, filled, kept, frac, over_text, note)

### Difficulty probe
Measures how badly page content competes with the page edge in the |Gx| profile.

In [ ]:
# ─────────────── difficulty probe (why a stage will struggle) ───────────────

@dataclass
class EdgeSignalStats:
    peak_to_interior: float      # >3 easy, <2 the old tracer cannot separate edge from content
    rival_columns: int           # columns rivalling the true edge's strength
    profile: np.ndarray


def edge_signal(img: np.ndarray) -> EdgeSignalStats:
    """Quantifies how badly page CONTENT competes with the page EDGE in the
    |Gx| column profile -- i.e. how hard this image is for a gradient-tracing
    detector.  This is the measurement that proved the operator's hypothesis:
    text-heavy median ratio 3.00 and image-heavy 2.92, versus 4.82 for
    diagram/sparse, with 2-4x the rival columns.  Kept in the pipeline as a
    per-image difficulty score so the scorecard can correlate failures with it.
    """
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).astype(np.float32)
    h, w = gray.shape
    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    prof = np.abs(gx[int(h * 0.35):int(h * 0.65), :]).mean(axis=0)
    prof = np.convolve(prof, np.ones(9) / 9, 'same')
    mx = float(prof.max()) if prof.size else 0.0
    rivals = int((prof > 0.5 * mx).sum())
    interior = prof[int(w * 0.2):int(w * 0.8)]
    med_int = float(np.median(interior)) if interior.size else 0.0
    ptr = mx / (med_int + 1e-6)
    return EdgeSignalStats(ptr, rivals, prof)

## 2. Run every stage over all 130 images

Pure stage functions, one pass, results kept in memory. ~0.12 s/image → ~15 s total,
which is fast enough to re-run after every threshold change.

In [ ]:
def run_all(files=FILES, work_w=WORK_W):
    out = []
    t0 = time.time()
    for n, f in enumerate(files):
        raw = cv2.imread(f)
        work, scale = to_work(raw, work_w)

        t = time.time()
        dsk = deskew(work)                       # stage 1
        pap = segment_paper(dsk.rotated)         # stage 2
        scp = check_scope(dsk.rotated, pap)      # scope gate
        bnd = detect_boundary(dsk.rotated, pap, scp) if scp.in_scope else None
        fng = detect_fingers(dsk.rotated, pap, bnd) if bnd else None
        cls = classify_content(dsk.rotated, pap.mask)
        sig = edge_signal(dsk.rotated)
        ms = (time.time() - t) * 1000

        out.append(dict(
            idx=n + 1, name=os.path.basename(f), path=f,
            work=work, scale=scale,
            deskew=dsk, paper=pap, scope=scp, bound=bnd, finger=fng,
            content=cls, signal=sig, ms=ms))
        if (n + 1) % 25 == 0:
            print(f'  ...{n+1}/{len(files)}', flush=True)
    print(f'done: {len(out)} images in {time.time()-t0:.1f}s')
    return out

R = run_all()

# Flatten the fields we plot repeatedly.
IDX      = [r['idx'] for r in R]
NAMES    = [f"#{r['idx']}" for r in R]
LABELS   = [r['content'].label for r in R]
ANGLES   = [r['deskew'].angle for r in R]
PAPER    = [r['paper'].area_frac for r in R]
RECTFILL = [r['paper'].rect_fill for r in R]
KIND     = [r['scope'].kind for r in R]
INSCOPE  = [r['scope'].in_scope for r in R]
WHITE    = [r['scope'].white_frac for r in R]
ASPECT   = [r['scope'].aspect for r in R]
SPAN     = [((r['bound'].right - r['bound'].left) / float(r['paper'].bbox[2]))
            if r['bound'] else 0.0 for r in R]
HASGUT   = [bool(r['bound'] and r['bound'].gutter is not None) for r in R]
BLOCKL   = [r['bound'].block_left if r['bound'] else 0 for r in R]
NFING    = [r['finger'].n_regions if r['finger'] else 0 for r in R]
FAREA    = [r['finger'].area_frac if r['finger'] else 0.0 for r in R]
OVERTEXT = [bool(r['finger'] and r['finger'].over_text) for r in R]
CONTRAST = [r['paper'].contrast for r in R]
PTR      = [r['signal'].peak_to_interior for r in R]
RIVALS   = [r['signal'].rival_columns for r in R]
MS       = [r['ms'] for r in R]

from collections import Counter
print()
print('content classes:', Counter(LABELS).most_common())
print('capture kinds  :', Counter(KIND).most_common())
print(f'per-image time: median {np.median(MS):.0f} ms')

## 3. The corpus — every image, grouped by content class

First look at what we are actually working with. Tiles are grouped so each content
class can be judged as a group.

In [ ]:
for cls in CLASS_ORDER:
    sel = [r for r in R if r['content'].label == cls]
    if not sel:
        continue
    grid([r['work'] for r in sel],
           [f"#{r['idx']}" for r in sel],
           cols=8, width=200,
           title=f'{cls}  —  {len(sel)} images')

## 4. Difficulty probe — why content type predicts failure

`peak/interior ratio` = strength of the strongest column gradient divided by the median
gradient inside the page. **High = the page edge stands out. Low = page content is as
strong as the edge**, and a gradient-tracing detector has nothing to lock onto.

This is the plot that justifies the whole v2 redesign.

In [ ]:
by_class(PTR, LABELS,
           title='Edge/content separability by content class  (higher = easier)',
           ylabel='peak / interior |Gx| ratio')

by_class(RIVALS, LABELS,
           title='Competing columns (>50% of peak strength)  — lower = easier',
           ylabel='rival column count')

hist(PTR, title='peak/interior ratio — whole corpus',
       xlabel='ratio',
       vlines=[(2.0, 'ratio 2.0 — tracer cannot separate', 'red'),
               (3.0, 'ratio 3.0 — marginal', 'orange')])

hard = [i for i, v in enumerate(PTR) if v < 2.0]
print(f'{len(hard)} images below ratio 2.0 (gradient tracing unreliable):',
      [R[i]['idx'] for i in hard])

In [ ]:
# The mechanism, on three images: an easy one, a hard one, and a dark cover.
fig, axes = plt.subplots(3, 2, figsize=(16, 11))
for row, idx in enumerate([43, 96, 17]):
    r = R[idx - 1]
    axes[row, 0].imshow(cv2.cvtColor(thumb(r['work'], 420), cv2.COLOR_BGR2RGB))
    axes[row, 0].axis('off')
    axes[row, 0].set_title(f"#{idx}  {r['content'].label}  "
                           f"ratio={r['signal'].peak_to_interior:.2f}")
    p = r['signal'].profile
    axes[row, 1].plot(p, lw=0.8)
    axes[row, 1].axhline(0.5 * p.max(), color='r', ls='--', lw=1,
                         label='50% of max')
    axes[row, 1].set_title('|Gx| column profile — what the old tracer walks on')
    axes[row, 1].legend(fontsize=8)
    axes[row, 1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Stage 1 results — Deskew

Ported from the operator's `bookcurve (8).ipynb`: Otsu → largest contour → `minAreaRect`
→ rotate. It keys on the **book silhouette**, not page content, so it is content-invariant
by construction — a blank page deskews as well as a text page.

Reference run: 130/130, range −16.66°…+15.33°, mean abs 2.68°.

**Known limitation:** `normalize_angle` maps to ±45°, so this corrects *skew*, not
*orientation* — it cannot tell 0° from 90°.

In [ ]:
hist(ANGLES, title='Stage 1 — detected skew angle, all 130',
       xlabel='degrees', bins=40)
by_class(np.abs(ANGLES), LABELS,
           title='|skew angle| by content class — should be FLAT if content-invariant',
           ylabel='|degrees|')

big = [abs(a) > 10 for a in ANGLES]
scorecard(LABELS, [not b for b in big], 'Stage 1 deskew (|angle| <= 10 deg)')

In [ ]:
# Every image, deskewed, with its angle. Flagged = large rotation, worth eyeballing.
grid([r['deskew'].rotated for r in R],
       [f"#{r['idx']}  {r['deskew'].angle:+.1f}°" for r in R],
       cols=8, width=200, flags=big,
       title='Stage 1 — deskewed output, ALL 130 (red = |angle| > 10°)')

In [ ]:
# Before/after for the largest rotations — the cases most likely to be wrong.
order = np.argsort(-np.abs(ANGLES))[:6]
fig, axes = plt.subplots(len(order), 2, figsize=(13, 3.4 * len(order)))
for row, i in enumerate(order):
    r = R[i]
    axes[row, 0].imshow(cv2.cvtColor(thumb(r['work'], 380), cv2.COLOR_BGR2RGB))
    axes[row, 0].set_title(f"#{r['idx']} original"); axes[row, 0].axis('off')
    axes[row, 1].imshow(cv2.cvtColor(thumb(r['deskew'].rotated, 380),
                                     cv2.COLOR_BGR2RGB))
    axes[row, 1].set_title(f"deskewed {r['deskew'].angle:+.2f}°")
    axes[row, 1].axis('off')
plt.tight_layout(); plt.show()

## 6. Stage 2 results — Paper segmentation

**The core design change.** Instead of "where is the strongest vertical gradient"
(which text defeats), we ask **"which pixels are paper"**. Paper stays bright where it
is printed on — dense text moves a neighbourhood's mean by tens of levels, while the
paper-to-stand step is ~185 levels here. So a region test is nearly content-invariant.

Thresholding is **Otsu on the whole frame**. A fixed offset from the corner background
was tried first and failed on ~26 of the 130: images 90–121 have a lit wall behind the
stand, so corner patches read ~47 while the frame median is ~150, and the wall floods
in as paper (measured paper fractions 0.77–0.98 where the book covers about half the
frame). Otsu splits on the image's own bimodal histogram and puts the wall on the dark
side where it belongs.

In [ ]:
hist(PAPER, title='Stage 2 — paper area as fraction of frame, all 130',
       xlabel='paper fraction',
       vlines=[(0.10, 'too small — under-segmented', 'red'),
               (0.75, 'too large — background leaked in', 'red')])

by_class(PAPER, LABELS,
           title='Paper fraction by content class — should be FLAT if content-invariant',
           ylabel='paper area fraction')

by_class(CONTRAST, LABELS,
           title='Paper-to-background contrast by content class',
           ylabel='levels')

# Shape, not just area. An open book nearly fills its own minAreaRect; a mask with a
# wall blob or hand fused on does not. This metric exists because area alone passed
# ~40 visibly contaminated masks -- the blob kept the area inside the accepted window.
hist(RECTFILL, title='Stage 2 - mask fill of its own minAreaRect (shape check)',
       xlabel='filled fraction',
       vlines=[(0.75, '0.75 - blob attached below this', 'red')])
by_class(RECTFILL, LABELS,
           title='Mask rectangularity by content class',
           ylabel='rect fill fraction')

# Out-of-scope captures (closed books) are excluded: their masks are SUPPOSED to be
# poor, so scoring them as segmentation failures would understate the stage.
paper_bad = [ins and (p < 0.10 or p > 0.75 or rf < 0.75)
             for p, rf, ins in zip(PAPER, RECTFILL, INSCOPE)]
scorecard(LABELS, [not b for b in paper_bad],
            'Stage 2 paper segmentation (0.10 <= fraction <= 0.75)')

In [ ]:
# ALL 130 with the paper mask tinted and its bounding box drawn.
ov = [draw_bbox(overlay_mask(r['deskew'].rotated, r['paper'].mask,
                                 (0, 0, 255), 0.40),
                  r['paper'].bbox, (0, 255, 0), 3) for r in R]
grid(ov, [f"#{r['idx']} {r['paper'].area_frac:.2f}" for r in R],
       cols=8, width=200, flags=paper_bad,
       title='Stage 2 — paper mask (red) + bbox (green), ALL 130')

In [ ]:
reasons = [r['paper'].note or
           ('paper fraction %.2f out of range' % r['paper'].area_frac)
           for r in R]
show_flagged(ov, NAMES, paper_bad, reasons, cols=3, width=520,
               title='Stage 2 FLAGGED — inspect these')

## 7. Scope gate — reject closed books, keep single sheets

Closed books (a photographed cover) are **out of scope**: no page to flatten, no gutter,
no split. Single-page documents (flyers, magazine covers, brochures) **stay in scope** —
they are genuinely *easier* than spreads, measuring `rect_fill` 0.88–0.997 against
0.95–0.99 for spreads. They simply take the no-gutter path.

The discriminator is the **fraction of the detected region that is near-white paper**,
because it keys on what the material physically *is*. Two simpler signals were tried and
both failed:

* **area fraction** — #43, a valid spread, sits at 0.226, right among the closed books;
* **region brightness** — #71/#73, valid single sheets, have median 102–104, *below*
  several closed books at 112–153.

White fraction separates cleanly: closed books 0.010–0.069, valid pages 0.157–0.924.
Aspect resolves the two edge cases — a full-bleed colour spread with no white margin
(#115, white 0.093) is accepted because it is landscape, and a dark cover where only the
title strip segments (#17/#18, aspect 3.77/2.48) is rejected because a real spread never
exceeds ~1.6.

In [ ]:
plt.figure(figsize=(9, 5.5))
cols = {'spread': '#2e7d32', 'single': '#1565c0', 'closed-book': '#c62828'}
for k, c in cols.items():
    ix = [i for i, kk in enumerate(KIND) if kk == k]
    if not ix:
        continue
    plt.scatter([WHITE[i] for i in ix], [ASPECT[i] for i in ix],
                c=c, label=f'{k} (n={len(ix)})', s=42, alpha=0.8,
                edgecolors='white', linewidths=0.5)
    for i in ix:
        if KIND[i] == 'closed-book':
            plt.annotate(f'#{IDX[i]}', (WHITE[i], ASPECT[i]),
                         fontsize=8, xytext=(4, 3), textcoords='offset points')
plt.axvline(0.12, color='red', ls='--', lw=1.2, label='white 0.12')
plt.axhline(1.10, color='gray', ls=':', lw=1.2, label='aspect 1.10 / 2.00')
plt.axhline(2.00, color='gray', ls=':', lw=1.2)
plt.xlabel('fraction of region that is near-white paper')
plt.ylabel('region aspect (w/h)')
plt.title('Scope gate — closed books separate on white fraction + aspect')
plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

scorecard(LABELS, INSCOPE, 'Scope gate (in-scope captures)')

In [ ]:
# Every rejected capture, shown large. These must ALL be closed books --
# a valid page appearing here is a false positive and a bug.
rej = [not s for s in INSCOPE]
show_flagged([r['work'] for r in R], NAMES, rej,
               [r['scope'].reason for r in R], cols=3, width=460,
               title='REJECTED as out of scope — verify every one is a closed book')

# And the single-sheet captures, which stay in scope on the no-gutter path.
sng = [k == 'single' for k in KIND]
grid([R[i]['work'] for i, b in enumerate(sng) if b],
       [f"#{R[i]['idx']}" for i, b in enumerate(sng) if b],
       cols=8, width=210,
       title='IN SCOPE — single-page documents (no gutter, no split)')

## 8. Stage 3 results — front-page boundary and gutter

The operator's requirement: bound the **printed left and right pages**, not the side
pages. Between the page and the stand sits the **fore-edge block** — the stack of
remaining leaves, which *is* paper and therefore inside the paper mask — and it must be
excluded.

Evidence is the per-column **mean brightness** of the paper region, not a gradient.
Content barely perturbs a column mean (it is a few levels of high-frequency noise on a
~220 plateau), which is exactly why this works where gradient tracing failed.

Two failures during development, both found by looking at the plots rather than the
numbers, and both worth recording:

1. **A plain longest-run returned one page, not the spread.** The gutter is a deep but
   *narrow* spike that cuts below the on-page threshold and splits the run in two — on
   #124 the result was `left=957 right=1491`, the right-hand page alone. Fixed by
   morphologically closing gaps up to 8% of the book width before taking the run: a
   gutter is tens of columns wide, the block is hundreds, so this bridges the spine
   without ever bridging the block. Gutter detection went from 14/108 to 96/108.
2. **Ink-heavy pages under-detected badly.** On full-bleed photo spreads the ink drags
   the column mean below "page level" and the run stops short — 37 of 124 in-scope
   captures, some down to span 0.20 of the paper bbox, every one with white fraction
   ≤ 0.75. Fixed by using Stage 2's paper mask extent as the outer bound (ink barely
   moves paper-vs-stand segmentation) and trimming inward only where a genuinely
   *darker plateau* is present, capped at 25% of width. Under-detection fell from 37 to 4.

In [ ]:
inb = [i for i, r in enumerate(R) if r['bound']]
spreads = [i for i in inb if KIND[i] == 'spread']
singles = [i for i in inb if KIND[i] == 'single']
print(f'gutter found on {sum(HASGUT[i] for i in spreads)}/{len(spreads)} spreads')
print(f'gutter wrongly found on {sum(HASGUT[i] for i in singles)}/{len(singles)} singles'
      '  (must be 0 -- a single sheet has no gutter)')

hist([SPAN[i] for i in inb],
       title='Stage 3 - detected page span as fraction of the paper bbox',
       xlabel='span / bbox width',
       vlines=[(0.80, '0.80 - under-detection below this', 'red')])
by_class([SPAN[i] for i in inb], [LABELS[i] for i in inb],
           title='Page span by content class - FLAT means content-invariant',
           ylabel='span / bbox width')

span_bad = [bool(r['bound']) and s < 0.80 for r, s in zip(R, SPAN)]
scorecard([LABELS[i] for i in inb], [not span_bad[i] for i in inb],
            'Stage 3 boundary (span >= 0.80 of paper bbox)')

In [ ]:
def draw_bounds(r):
    o = r['deskew'].rotated.copy()
    b = r['bound']
    if b is None:
        return o
    cv2.line(o, (b.left, b.top), (b.left, b.bottom), (0, 255, 0), 4)
    cv2.line(o, (b.right, b.top), (b.right, b.bottom), (0, 255, 0), 4)
    if b.gutter is not None:
        cv2.line(o, (b.gutter, b.top), (b.gutter, b.bottom), (255, 0, 255), 4)
    return o

ovb = [draw_bounds(r) for r in R]
grid([ovb[i] for i in inb],
       [f"#{R[i]['idx']} {'g' if HASGUT[i] else '-'} {SPAN[i]:.2f}" for i in inb],
       cols=8, width=200, flags=[span_bad[i] for i in inb],
       title='Stage 3 - green = page edges, magenta = gutter, ALL in-scope captures')

In [ ]:
# The under-detected cases, large. Green lines cutting into the printed page here
# means the boundary is wrong and needs work -- not a metric to be tuned away.
show_flagged(ovb, NAMES, span_bad,
               [f'span {SPAN[i]:.2f} of bbox' for i in range(len(R))],
               cols=3, width=520, title='Stage 3 FLAGGED - under-detected span')

# Spreads where no gutter was found. Some are genuinely flat-lying books with no
# spine shadow; any that clearly show a spine are a miss.
nogut = [KIND[i] == 'spread' and not HASGUT[i] for i in range(len(R))]
show_flagged(ovb, NAMES, nogut,
               [R[i]['bound'].note if R[i]['bound'] else '' for i in range(len(R))],
               cols=3, width=520, title='Stage 3 - spreads with NO gutter detected')

## 9. Stage 4 results — finger mask and page-colour fill

The requirement: **fill the finger region with the colour of the page.** Deliberately not
content reconstruction — nothing is invented that could pass as real text. A flat fill
either looks right (on a margin, the dominant case here: thumbs holding the book open) or
looks obviously like a patch, which is a safe failure.

Two things make this work that the old pipeline's skin detector did not have:

1. **Stage 3 already knows where the page is**, so only skin *inside* the page bounds is
   considered — a hand resting on the stand is irrelevant.
2. **Printed photos of people are rejected structurally.** A real finger enters from
   outside the page, so its region must touch the page boundary. A skin-toned region fully
   enclosed by page content never does, however finger-shaped it looks.

The chroma window had to be tightened after measuring: textbook YCrCb bounds
(Cr 135–180, Cb 85–135) fired on **105 of 124** captures with regions up to **53% of the
page** — aged cream book paper sits close to skin in chroma. Adding a luma test (skin is
markedly darker than lit paper, ratio < 0.82) and an area cap brought this to 69/124
(56%), median region 1.3% of page area, which matches the visible rate in the corpus.

The fill colour is sampled **locally** from surrounding page pixels, not as one global
page colour: a curved page shades noticeably from gutter to fore-edge, so a single flat
value would show as a mismatched rectangle.

In [ ]:
fin = [i for i, r in enumerate(R) if r['finger']]
print(f'fingers detected on {sum(1 for i in fin if NFING[i])} / {len(fin)} in-scope captures')
print(f'regions overlapping inked content: {sum(OVERTEXT)}')

hist([FAREA[i] for i in fin if NFING[i]],
       title='Stage 4 - finger region as fraction of page area',
       xlabel='area fraction',
       vlines=[(0.22, '0.22 - rejected above this', 'red')])
by_class([FAREA[i] for i in fin], [LABELS[i] for i in fin],
           title='Finger area by content class - IMAGE-HEAVY is the false-positive risk',
           ylabel='area fraction')

In [ ]:
# Mask overlay for every capture where a finger was found.
fmask = [i for i in fin if NFING[i]]
grid([overlay_mask(R[i]['deskew'].rotated, R[i]['finger'].mask, (0, 0, 255), 0.55)
        for i in fmask],
       [f"#{R[i]['idx']} {FAREA[i]:.3f}{' TEXT' if OVERTEXT[i] else ''}" for i in fmask],
       cols=8, width=200, flags=[OVERTEXT[i] for i in fmask],
       title='Stage 4 - detected finger mask (red). Red border = overlaps inked content')

In [ ]:
# Before / after fill, on the largest detections -- this is where a bad mask shows.
top = sorted(fmask, key=lambda i: -FAREA[i])[:8]
fig, axes = plt.subplots(len(top), 2, figsize=(13, 3.1 * len(top)))
for row, i in enumerate(top):
    r = R[i]
    axes[row, 0].imshow(cv2.cvtColor(thumb(r['deskew'].rotated, 420), cv2.COLOR_BGR2RGB))
    axes[row, 0].set_title(f"#{r['idx']} original"); axes[row, 0].axis('off')
    axes[row, 1].imshow(cv2.cvtColor(thumb(r['finger'].filled, 420), cv2.COLOR_BGR2RGB))
    axes[row, 1].set_title(f"filled with page colour ({FAREA[i]:.1%} of page)"
                           + ('  -- OVER TEXT' if OVERTEXT[i] else ''))
    axes[row, 1].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# Fingers sitting on inked content: the fill ERASES that text. Shown so the
# operator can decide policy (fill anyway vs flag for recapture) with the real
# cases in front of them rather than in the abstract.
show_flagged([r['finger'].filled if r['finger'] else r['work'] for r in R],
               NAMES, OVERTEXT,
               [r['finger'].note if r['finger'] else '' for r in R],
               cols=3, width=520,
               title='Stage 4 - finger over inked content (fill erases it)')

## 10. Scorecard

Where we stand **before** any boundary/gutter/finger work. Every later change is measured
against this table.

In [ ]:
print('=' * 64)
print('PHASE 0 BASELINE'.center(64))
print('=' * 64)
scorecard(LABELS, [not b for b in big],      'Stage 1  deskew')
scorecard(LABELS, [not b for b in paper_bad],'Stage 2  paper segmentation')
scorecard([LABELS[i] for i in inb], [not span_bad[i] for i in inb],
            'Stage 3  front-page boundary')
scorecard([LABELS[i] for i in fin], [not OVERTEXT[i] for i in fin],
            'Stage 4  finger fill (safe = not over inked content)')

print(f"\nper-image time: median {np.median(MS):.0f} ms  "
      f"(full corpus {np.sum(MS)/1000:.1f}s)")
print(f"skew angle    : {np.min(ANGLES):+.2f}° .. {np.max(ANGLES):+.2f}°  "
      f"mean|.| {np.mean(np.abs(ANGLES)):.2f}°")
print(f"paper fraction: {np.min(PAPER):.3f} .. {np.max(PAPER):.3f}  "
      f"median {np.median(PAPER):.3f}")

rows = [dict(idx=r['idx'], name=r['name'], label=r['content'].label,
             angle=round(r['deskew'].angle, 2),
             paper=round(r['paper'].area_frac, 4),
             contrast=round(r['paper'].contrast, 1),
             rect_fill=round(r['paper'].rect_fill, 4),
             ptr=round(r['signal'].peak_to_interior, 3),
             rivals=r['signal'].rival_columns,
             kind=r['scope'].kind, in_scope=r['scope'].in_scope,
             white=round(r['scope'].white_frac, 4),
             aspect=round(r['scope'].aspect, 3),
             span=round(SPAN[R.index(r)], 4),
             gutter=(r['bound'].gutter if r['bound'] else None),
             page_left=(r['bound'].left if r['bound'] else None),
             page_right=(r['bound'].right if r['bound'] else None),
             note=r['paper'].note, ms=round(r['ms'], 1)) for r in R]
with open('phase0_baseline.json', 'w') as fh:
    json.dump(rows, fh, indent=1)
print('\nwrote phase0_baseline.json')

### Known open issues at end of Phase 0

Recorded here so they are not silently carried forward:

0. **Stage 3 open items.** Four captures of one dark magazine cover (#58-61) trim ~20%
   off the right edge -- the cover's own dark right side reads as a "darker plateau" and
   is mistaken for a fore-edge block. Five flat-lying spreads (#24, 26, 28, 35, 77) report
   no gutter, which is the honest answer: their deepest spine dip is 7-13 levels against
   a 13-level threshold. Boundaries on all five are correct.
0. **Closed books are now rejected up front** by the scope gate (#17, 18, 30–33), with
   zero valid pages wrongly skipped. Single-page documents stay in scope on a no-gutter
   path.
1. **Residual wall blob on ~6 images** (#109, #112, #117, #121–123). The occupancy trim
   removed it from the majority (#97–107, #113, #119, #124–130 are now clean), but where
   the blob overlaps the book's own row band it survives both the occupancy trim and
   `rect_fill` (min is now 0.832, so nothing flags). Needs a left-edge verticality test in
   Stage 3 — the book's true side edge is a long straight near-vertical line, the blob's is
   not.
3. **The fore-edge block is included in the paper mask.** Correct at this stage: the mask
   is *paper*, and the block *is* paper. Separating the front page from the block is
   Stage 3's job (the "front pages, not side pages" requirement).

### A metric lesson from Phase 0

The first version of this stage scored **98% pass on area fraction alone**, and the
contact sheet showed that was wrong: a lit wall behind the stand was fused into roughly
**40** of the masks (images ~90–130). The blob kept the area inside the accepted window,
so the metric never fired, and because the wall is *connected* to the book, taking the
largest component could not drop it either.

Two fixes came out of that, both kept:

* the paper threshold is now anchored on the **paper mode** (`bg + 0.62*contrast`, floored
  at Otsu) rather than on Otsu alone — paper sits at 220–240 while the wall sits at
  90–118, a 116+ level gap on every image measured, whereas Otsu was landing at 126–148
  and cutting *below* the wall;
* a **shape** metric (`rect_fill`, how much of its own `minAreaRect` the mask fills) now
  runs beside the area metric — an open book is nearly a filled rectangle;
* and a **column/row occupancy trim**: a wall or hand blob occupies ~0.17–0.19 of a
  column's height while a real book column occupies ~0.7+, so keeping only the run of
  columns containing the occupancy peak separates them structurally rather than by
  threshold. This is what finally removed most of the blobs; raising the brightness
  threshold alone shrank them but could not remove them, because the wall is *connected*
  to the book and convex enough that `rect_fill` caught only one.

The general point, which applies to every stage after this one: **an aggregate metric can
pass while the picture is plainly wrong.** Every stage gets a contact sheet of all 130 for
this reason, and a stage is not "done" until the sheet has actually been looked at.

### Next — Phase 1: front-page boundary

Search **inward** from the paper mask's border to find the printed page edge, rather
than globally for the strongest gradient. The mask constrains the search band, which is
what removes the 644 competing columns on image 43.